### 主成分分析(PCA)とは
- データの次元を削減する手法。
- PCAはデータの分散が最大になる方向（主成分）を見つけ、その方向に射影することで次元を圧縮する。
- 情報の損失を最小限に抑えながら次元を削減できるのが特徴。

||内容|
|----|----|
|手法|教師なし学習（次元削減）|
|目的|分散が最大になる方向への射影|
|解法|共分散行列の固有値問題|
|主成分|固有値が大きい順の固有ベクトル|
|寄与率|各主成分が説明する分散の割合|


In [ ]:
# ライブラリでの実装
# 4次元データを2次元に削減する例

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA

iris = load_iris()
X, y = iris.data, iris.target

model = PCA(n_components=2)
X_reduced = model.fit_transform(X)

print(f"元の形状: {X.shape}")
print(f"削減後の形状: {X_reduced.shape}")
print(f"寄与率: {model.explained_variance_ratio_}")
print(f"累積寄与率: {model.explained_variance_ratio_.sum():.4f}")

クラスター割り当て（先頭10件）: [1 1 1 1 1 1 1 1 1 1]
正解ラベル（先頭10件）        : [0 0 0 0 0 0 0 0 0 0]


In [ ]:
# scratchでのPCA実装

import numpy as np
from sklearn.datasets import load_iris


class PCA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.explained_variance_ratio_ = None
        self.mean = None

    def fit(self, X):
        self.mean = X.mean(axis=0)
        X_centered = X - self.mean

        S = X_centered.T @ X_centered / len(X)

        eigenvalues, eigenvectors = np.linalg.eigh(S)

        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        self.components = eigenvectors[:, :self.n_components].T
        total_var = eigenvalues.sum()
        self.explained_variance_ratio_ = eigenvalues[:self.n_components] / total_var

    def transform(self, X):
        X_centered = X - self.mean
        return X_centered @ self.components.T

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


# 実行
iris = load_iris()
X, y = iris.data, iris.target

model = PCA(n_components=2)
X_reduced = model.fit_transform(X)

print(f"元の形状: {X.shape}")
print(f"削減後の形状: {X_reduced.shape}")
print(f"寄与率: {model.explained_variance_ratio_}")
print(f"累積寄与率: {model.explained_variance_ratio_.sum():.4f}")



クラスター割り当て（先頭10件）: [1 1 1 1 1 1 1 1 1 1]
正解ラベル（先頭10件）        : [0 0 0 0 0 0 0 0 0 0]


`np.linalg.eigh`

対称行列（共分散行列は必ず対称）の固有値・固有ベクトルを求めるときは `eigh` を使います。`eig` より数値的に安定しており、固有値が実数で返ることが保証されます。

固有ベクトルの並び替え

`eigh` は固有値を昇順で返すため、`np.argsort(eigenvalues)[::-1]` で降順に並び替えます。対応する固有ベクトルも同じ順に並び替えます。

`self.components`

上位 `n_components` 個の固有ベクトルを行方向に並べた行列です。`transform` では `X_centered @ self.components.T` で射影を計算します。